# Emotion-Based Talking Head Generation (Standalone)

Generates **emotion-controlled talking head videos** with IMTalker using an **already-trained
emotion LoRA adapter** downloaded from Hugging Face:
[`bitwisemind/emotion_lora_adapter_v1`](https://huggingface.co/bitwisemind/emotion_lora_adapter_v1)
(file `last.ckpt`).

**This repository is fully self-contained** -- all required source code (`generator/`,
`renderer/`) is bundled, so no external repository is needed. It reuses the exact avatar
testing / video generation method that runs after each training epoch in
`05_train_emotion_lora.ipynb` from the full training pipeline -- but **without any LoRA
training** and without the real-time PersonaPlex conversation pipeline.

**Quick start on RunPod**
1. `git clone https://github.com/MoshiHead/emotion_talking_head_test_only.git`
2. `cd emotion_talking_head_test_only` and open this notebook
3. Run all cells -- models download automatically, then the video is generated

**Pipeline**
1. Verify the bundled source code and locate the repository root
2. Set up a dedicated virtual environment (CUDA PyTorch + dependencies) -- the same
   venv + subprocess architecture as the live-pipeline notebooks, so the kernel never
   imports numpy/torch and **no kernel restart is ever needed**
3. Download and configure all required models:
   - Base IMTalker generator / renderer / wav2vec2 (`cbsjtu01/IMTalker`)
   - Style LoRA merged into the base generator (`niloy629/hdtf_preprocess`) -- matches the
     emotion adapter training setup
   - Emotion LoRA adapter `last.ckpt` (`bitwisemind/emotion_lora_adapter_v1`)
4. Run `generator/test_emotions.py` inside the venv (base generator -> style LoRA merge
   -> emotion LoRA adapters -> emotion checkpoint -> generate the video)
5. Display the generated talking head video(s) inline

**Requirements**
- Any CUDA GPU (RTX 3090/4090/5090, A100, A40, A6000, L40S, H100, ...)
- `ffmpeg` available on `PATH` (preinstalled on RunPod images)
- ~8 GB of disk space (venv with CUDA PyTorch + model downloads); venv and checkpoints
  persist on RunPod volumes

**Emotions**: `neutral` (0), `happy` (1), `sad` (2), `angry` (3), `surprise` (4), `fear` (5)

In [ ]:
!git clone https://github.com/MoshiHead/emotion_talking_head_test_only.git
%cd emotion_talking_head_test_only

In [ ]:
# ============================================================================
# 1. Locate the repository root (the folder containing generator/ and renderer/)
# ============================================================================
from pathlib import Path


def find_repo_root():
    """Walk up from the current working directory to find the repo root."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "generator" / "FM.py").exists() and (cand / "renderer" / "models.py").exists():
            return cand
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside the "
        "cloned repository folder (the folder that contains generator/ and renderer/)."
    )


REPO_ROOT = find_repo_root()

required_files = [
    "generator/FM.py",
    "generator/FMT.py",
    "generator/wav2vec2.py",
    "generator/generate.py",
    "generator/train_lora.py",
    "generator/options/base_options.py",
    "renderer/models.py",
    "renderer/modules.py",
    "renderer/attention_modules.py",
    "renderer/lia_resblocks.py",
]
for rel in required_files:
    assert (REPO_ROOT / rel).exists(), f"Missing required file: {rel}"

print(f"Repository root: {REPO_ROOT}")
print(f"Verified {len(required_files)} required source files")
print("Code structure OK -- no external repository needed")

In [ ]:
# ============================================================================
# 2. CONFIGURATION -- EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Inputs (EDIT ME)
# ---------------------------------------------------------------------------
# Reference image: portrait / avatar face (resized to 512x512)
REF_IMAGE = str(REPO_ROOT / "assets" / "3robert.jpeg")

# Input audio: .wav speech file (16 kHz mono recommended)
AUDIO = str(REPO_ROOT / "assets" / "audio_3.wav")

# Emotion name: "neutral", "happy", "sad", "angry", "surprise", "fear"
# (also accepts an integer 0-5, or "all" to render all 6 emotions)
EMOTION = "happy"

# ---------------------------------------------------------------------------
# Generation settings (same defaults as the per-epoch training tests)
# ---------------------------------------------------------------------------
CFG_SCALE = 3.0     # audio classifier-free guidance scale
NFE       = 10      # number of flow-matching function evaluations
SEED      = 42      # random seed
CROP      = False   # False = plain 512x512 resize (safe for robot faces, no face detection)

# Where generated videos are saved
OUTPUT_DIR = str(REPO_ROOT / "emotion_talking_head_outputs")

# ---------------------------------------------------------------------------
# Emotion LoRA adapter (downloaded from Hugging Face)
# ---------------------------------------------------------------------------
HF_EMOTION_LORA_REPO = "bitwisemind/emotion_lora_adapter_v1"
HF_EMOTION_LORA_FILE = "last.ckpt"
EMOTION_LORA_RANK  = 32     # must match training (LORA_RANK in 05_train_emotion_lora.ipynb)
EMOTION_LORA_ALPHA = 64.0   # must match training (LORA_ALPHA)

# ---------------------------------------------------------------------------
# Optional style LoRA merged into the base generator before the emotion
# adapter is applied (matches how the emotion adapter was trained)
# ---------------------------------------------------------------------------
USE_STYLE_LORA     = True
HF_STYLE_LORA_REPO = "niloy629/hdtf_preprocess"    # dataset repository
HF_STYLE_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
STYLE_LORA_RANK    = 64
STYLE_LORA_ALPHA   = 128.0

# ---------------------------------------------------------------------------
# Derived paths (no need to edit) -- all downloads stay inside this repo folder
# ---------------------------------------------------------------------------
CKPT_DIR            = REPO_ROOT / "checkpoints"
WAV2VEC_DIR         = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT       = CKPT_DIR / "renderer.ckpt"
BASE_GENERATOR_CKPT = CKPT_DIR / "generator.ckpt"
STYLE_LORA_CKPT     = CKPT_DIR / HF_STYLE_LORA_FILE
EMOTION_LORA_CKPT   = CKPT_DIR / "emotion_lora" / HF_EMOTION_LORA_FILE

EMOTION_MAP   = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}


def resolve_emotions(spec):
    """Accepts a name ('happy'), an integer (1), or 'all'."""
    if isinstance(spec, int):
        return [spec]
    s = str(spec).strip().lower()
    if s == "all":
        return [0, 1, 2, 3, 4, 5]
    if s.isdigit():
        return [int(s)]
    assert s in EMOTION_MAP, f"Unknown emotion {spec!r}. Valid: {list(EMOTION_MAP)} or 'all'"
    return [EMOTION_MAP[s]]


EMOTION_IDS = resolve_emotions(EMOTION)

print(f"reference image : {REF_IMAGE}")
print(f"audio           : {AUDIO}")
print(f"emotion(s)      : {[EMOTION_NAMES[i] for i in EMOTION_IDS]} (ids={EMOTION_IDS})")
print(f"generation      : cfg={CFG_SCALE}  nfe={NFE}  seed={SEED}  crop={CROP}")
print(f"output dir      : {OUTPUT_DIR}")
print(f"emotion adapter : {HF_EMOTION_LORA_REPO}/{HF_EMOTION_LORA_FILE} "
      f"(rank={EMOTION_LORA_RANK}, alpha={EMOTION_LORA_ALPHA})")
print(f"style LoRA      : {HF_STYLE_LORA_FILE if USE_STYLE_LORA else '(disabled)'}")

In [ ]:
# ============================================================================
# 3. Environment setup -- dedicated venv (CUDA PyTorch + dependencies)
#
#    Same architecture as the live-pipeline notebooks (06 / RunPod_RTX5090):
#    everything is installed into a SEPARATE virtual environment (venv/) and
#    all heavy work runs through the venv Python as a subprocess. The notebook
#    kernel itself NEVER imports numpy / torch / cv2, so a top-to-bottom run
#    NEVER needs a kernel restart.
# ============================================================================
import importlib.util, json as _json, os, shutil, subprocess, sys

VENV_DIR = REPO_ROOT / "venv"
VENV_PYTHON = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def _venv_pip(*args):
    print("[venv pip]", " ".join(a for a in args if not a.startswith("http")))
    subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-q"] + list(args), check=True)


def _venv_run(code):
    """Run python code inside the venv; returns the CompletedProcess."""
    return subprocess.run([str(VENV_PYTHON), "-c", code], capture_output=True, text=True)


def _venv_pkg_version(name):
    code = ("import importlib.metadata as m\n"
            "try: print(m.version(" + repr(name) + "))\n"
            "except m.PackageNotFoundError: print()")
    return _venv_run(code).stdout.strip()


# --- 1. Create the venv ----------------------------------------------------------
if not VENV_PYTHON.exists():
    print(f"[venv] creating virtual environment at {VENV_DIR} ...")
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
    _venv_pip("--upgrade", "pip", "setuptools", "wheel")
else:
    print(f"[venv] reusing existing virtual environment at {VENV_DIR}")


# --- 2. CUDA-capable PyTorch inside the venv -------------------------------------
def venv_torch_ok():
    res = _venv_run("import torch; print(torch.cuda.is_available())")
    return res.returncode == 0 and res.stdout.strip() == "True"


# Pinned to the proven stack used by ALL reference notebooks and live.md:
# torch 2.8.0 + torchvision 0.23.0 (cu128 wheels cover sm_75..sm_120: RTX
# 3090/4090/5090, A100, A40, A6000, L40S, H100). Do NOT install unpinned latest:
# newer torchvision releases removed torchvision.io.write_video entirely.
TORCH_PIN = ("torch==2.8.0", "torchvision==0.23.0")

if not venv_torch_ok():
    print("[venv] CUDA-capable PyTorch not found -- installing pinned torch 2.8.0 + torchvision 0.23.0")
    print("       (first run downloads ~3 GB) ...")
    installed = False
    for index_url in [
        "https://download.pytorch.org/whl/cu128",
        "https://download.pytorch.org/whl/cu126",   # fallback
    ]:
        try:
            print(f"[venv] installing ({index_url.rsplit(chr(47), 1)[-1]} wheels) ...")
            _venv_pip(*TORCH_PIN, "--index-url", index_url)
            if venv_torch_ok():
                installed = True
                break
        except subprocess.CalledProcessError:
            print("  wheel index failed, trying next option ...")
    if not installed:
        _venv_pip(*TORCH_PIN)   # default PyPI wheels
    assert venv_torch_ok(), "CUDA still unavailable inside the venv after installing PyTorch."
else:
    # An existing working venv is kept as-is (no re-download). Video encoding no
    # longer depends on torchvision.io.write_video, so any torch version works.
    print("[venv] CUDA-capable PyTorch already installed -- keeping it (no re-download)")

# GPU info (queried inside the venv, like the reference notebooks)
probe = _venv_run(
    "import torch, json; print(json.dumps({"
    "'torch': torch.__version__,"
    "'gpu': torch.cuda.get_device_name(0),"
    "'cc': '.'.join(map(str, torch.cuda.get_device_capability(0))),"
    "'vram': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)}))"
)
assert probe.returncode == 0, probe.stderr
gpu_info = _json.loads(probe.stdout.strip().splitlines()[-1])
print("torch (venv):       ", gpu_info["torch"])
print("GPU:                ", gpu_info["gpu"])
print("Compute capability: ", gpu_info["cc"])
print("VRAM (GB):          ", gpu_info["vram"])
assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (required for video encoding)."

# --- 3. Python dependencies (installed INTO THE VENV) -----------------------------
required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",
    "torchdiffeq":       "torchdiffeq==0.2.5",
    "einops":            "einops",
    "transformers":      "transformers>=4.38,<4.42",
    "librosa":           "librosa",
    "face_alignment":    "face_alignment==1.4.1",
    "av":                "av==12.0.0",
    "tensorboard":       "tensorboard",
    "timm":              "timm==1.0.9",
    "tqdm":              "tqdm",
    "huggingface_hub":   "huggingface_hub",
    "yaml":              "pyyaml",
    "cv2":               "opencv-python<4.10.0.0",
    "matplotlib":        "matplotlib<3.9.0",
}
mods_probe = _venv_run(
    "import importlib.util, json\n"
    "print(json.dumps([m for m in " + _json.dumps(list(required)) +
    " if importlib.util.find_spec(m) is None]))"
)
missing = [required[m] for m in _json.loads(mods_probe.stdout.strip() or "[]")]
if missing:
    _venv_pip(*missing)
else:
    print("[venv] all required packages already installed")

# --- 4. NumPy / SciPy pins (inside the venv) ---------------------------------------
# cv2 and other compiled modules in this pipeline are built against NumPy 1.x,
# so numpy must stay <2.0.0 and scipy <1.13 (the last NumPy-1.x-compatible line).
np_ver = _venv_pkg_version("numpy") or "0"
if int(np_ver.split(".")[0]) >= 2:
    print(f"[venv] numpy {np_ver} detected -- pinning numpy<2.0.0 ...")
    _venv_pip("numpy<2.0.0")
    np_ver = _venv_pkg_version("numpy")
print(f"numpy (venv):       {np_ver}")

sc_ver = _venv_pkg_version("scipy")
sc_mm = tuple(int(x) for x in sc_ver.split(".")[:2] if x.isdigit()) if sc_ver else ()
if not sc_mm or sc_mm >= (1, 13):
    print("[venv] scipy", sc_ver or "(missing)", "-- installing scipy<1.13 (NumPy-1.x compatible) ...")
    _venv_pip("scipy<1.13")
    sc_ver = _venv_pkg_version("scipy")
print(f"scipy (venv):       {sc_ver}")

# --- 5. Verify the full import stack INSIDE THE VENV --------------------------------
stack_code = ("import numpy, cv2, scipy.integrate, librosa, matplotlib.pyplot, "
              "face_alignment, torchdiffeq")
stack_test = _venv_run(stack_code)
if stack_test.returncode != 0:
    err = stack_test.stderr
    print("[venv] import stack check failed -- attempting targeted repairs ...")
    if "numba" in err:
        _venv_pip("numba<0.62")                         # NumPy 1.x-compatible numba
    if "scipy" in err:
        _venv_pip("scipy<1.13")
    if "cv2" in err or "numpy.core.multiarray" in err:
        _venv_pip("--force-reinstall", "--no-deps", "opencv-python<4.10.0.0")
    stack_test = _venv_run(stack_code)
if stack_test.returncode != 0:
    raise RuntimeError("Venv import stack still failing:\n" + stack_test.stderr.strip()[-600:])
print("[venv] import stack check: OK (numpy, cv2, scipy.integrate, librosa, "
      "matplotlib.pyplot, face_alignment, torchdiffeq)")

# --- 6. huggingface_hub in the KERNEL env (pure Python -- no numpy) ------------------
# Cell 4 uses hf_hub_download from the kernel; everything else runs in the venv.
if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

print("Environment OK (venv) -- the kernel never imports numpy/torch, so no restart is needed")

In [ ]:
# ============================================================================
# 4. Download all required models & checkpoints
# ============================================================================
from huggingface_hub import hf_hub_download

CKPT_DIR.mkdir(parents=True, exist_ok=True)


def download(repo_id, filename, local_dir, repo_type="model"):
    dest = Path(local_dir) / filename
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {dest}")
        return dest
    print(f"[download] {filename} from {repo_id} ...")
    hf_hub_download(repo_id=repo_id, filename=filename,
                    local_dir=str(local_dir), repo_type=repo_type)
    print(f"    ready: {dest}")
    return dest


# 1) Base IMTalker models (config, renderer, full base generator)
for fname in ["config.yaml", "renderer.ckpt", "generator.ckpt"]:
    download("cbsjtu01/IMTalker", fname, CKPT_DIR)

# 2) wav2vec2-base-960h audio encoder
for rel in [
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]:
    download("cbsjtu01/IMTalker", rel, CKPT_DIR)

# 3) Style LoRA (merged into the base generator, matching emotion-adapter training)
if USE_STYLE_LORA:
    download(HF_STYLE_LORA_REPO, HF_STYLE_LORA_FILE,
             CKPT_DIR, repo_type="dataset")

# 4) Trained emotion LoRA adapter from Hugging Face
download(HF_EMOTION_LORA_REPO, HF_EMOTION_LORA_FILE, EMOTION_LORA_CKPT.parent)

# Final verification
for p in [RENDERER_CKPT, BASE_GENERATOR_CKPT,
          WAV2VEC_DIR / "pytorch_model.bin", EMOTION_LORA_CKPT]:
    assert Path(p).exists(), f"Missing required file: {p}"
if USE_STYLE_LORA:
    assert STYLE_LORA_CKPT.exists(), f"Missing style LoRA: {STYLE_LORA_CKPT}"

print("\nAll models ready:")
print(f"  base generator : {BASE_GENERATOR_CKPT}")
print(f"  renderer       : {RENDERER_CKPT}")
print(f"  wav2vec2       : {WAV2VEC_DIR}")
if USE_STYLE_LORA:
    print(f"  style LoRA     : {STYLE_LORA_CKPT}")
print(f"  emotion LoRA   : {EMOTION_LORA_CKPT}")

In [ ]:
# ============================================================================
# 5. Generate the emotion talking-head video(s)
#    Runs the bundled generator/test_emotions.py (the same proven code the
#    per-epoch training tests use) inside the venv as a subprocess.
# ============================================================================
import subprocess

assert VENV_PYTHON.exists(), "venv missing -- run cell 3 first"
assert EMOTION_LORA_CKPT.exists(), (
    f"Emotion LoRA checkpoint missing: {EMOTION_LORA_CKPT} -- run cell 4 first")

cmd = [
    str(VENV_PYTHON), "generator/test_emotions.py",
    "--checkpoint", str(EMOTION_LORA_CKPT),
    "--renderer_ckpt", str(RENDERER_CKPT),
    "--base_generator_ckpt", str(BASE_GENERATOR_CKPT),
    "--wav2vec_model_path", str(WAV2VEC_DIR),
    "--ref_img", REF_IMAGE,
    "--audio", AUDIO,
    "--out_dir", OUTPUT_DIR,
    "--cfg", str(CFG_SCALE),
    "--nfe", str(NFE),
    "--seed", str(SEED),
    "--lora_rank", str(EMOTION_LORA_RANK),
    "--lora_alpha", str(EMOTION_LORA_ALPHA),
    "--emotions", *[EMOTION_NAMES[i] for i in EMOTION_IDS],
]
if USE_STYLE_LORA:
    cmd += ["--style_lora_ckpt", str(STYLE_LORA_CKPT)]
if CROP:
    cmd += ["--crop"]

print("$ " + " ".join(cmd) + "\n")
proc = subprocess.Popen(
    cmd, cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(
        f"Generation failed with exit code {proc.returncode} (see output above)")

generated = [Path(OUTPUT_DIR) / f"{i}_{EMOTION_NAMES[i]}.mp4" for i in EMOTION_IDS]
for f in generated:
    assert f.exists() and f.stat().st_size > 0, f"Expected output missing: {f}"
print(f"\n{len(generated)} video(s) ready in {Path(OUTPUT_DIR).resolve()}")

In [ ]:
# ============================================================================
# 6. Display the generated video(s) inline
# ============================================================================
from IPython.display import Video, display

for f in generated:
    print(f.name)
    display(Video(str(f), embed=True, width=512))

## Notes & Troubleshooting

- **No kernel restarts**: the environment cell (3) creates a dedicated virtual
  environment (`venv/`) and all heavy work runs through its Python as a subprocess
  (the same architecture as the live-pipeline notebooks `06_emotion_live_pipeline.ipynb`
  and `RunPod_RTX5090_PersonaPlex_IMTalker_Live_fixed.ipynb`). The notebook kernel never
  imports numpy / torch, so a top-to-bottom run never needs a kernel restart -- even on
  a NumPy 2.x base image.
- **First run**: creating the venv downloads PyTorch (~3 GB) plus dependencies -- a few
  minutes, only once. The venv and downloaded checkpoints persist on RunPod volumes, so
  later runs skip straight to generation.
- **Change emotion / inputs**: edit the CONFIGURATION cell (2), then re-run the Generate
  cell (5). The model reloads each run (same behaviour as the CLI script).
- **Rank / alpha**: `EMOTION_LORA_RANK = 32` and `EMOTION_LORA_ALPHA = 64.0` must match the
  values used when the adapter was trained (`05_train_emotion_lora.ipynb`).
- **Robot / non-human faces**: keep `CROP = False` (plain 512x512 resize, no face detection).
  Set `CROP = True` to enable face-alignment cropping for human faces.
- **Black frames**: almost always means the full base `generator.ckpt` was not downloaded
  correctly -- re-run the download (4) and generate (5) cells.
- **Style LoRA**: the emotion adapter was trained on top of the base generator with the
  style LoRA merged in; keep `USE_STYLE_LORA = True` to reproduce training-time behaviour.
- **All emotions**: set `EMOTION = "all"` to render `0_neutral.mp4` ... `5_fear.mp4` in one run
  (same behaviour as `generator/test_emotions.py` without the `--emotions` filter).
- **CLI alternative**: the bundled `generator/test_emotions.py` also renders videos from
  the terminal, e.g. `venv/bin/python generator/test_emotions.py --checkpoint checkpoints/emotion_lora/last.ckpt
  --ref_img assets/3robert.jpeg --audio assets/audio_3.wav --out_dir outputs --cfg 3.0`